<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-38.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 38 - Evaluación SBD-RA3: Ejercicios de repaso

**Módulo:** Sistemas de Big Data (SBD - 5074)

**Bloque:** RA3 - Gestión y almacenamiento

---

## Objetivo

Este notebook contiene ejercicios de práctica tipo examen para preparar la evaluación
del RA3. Cubre los temas principales:

1. Formatos de almacenamiento y comparación
2. Operaciones SQL y NoSQL
3. Integración de datos (simulando Flume/Sqoop)
4. Diseño de pipelines
5. R vs Python

In [ ]:
import pandas as pd
import numpy as np
import json
import sqlite3
import os
import time
from datetime import datetime, timedelta

print("=== Entorno de evaluación cargado ===")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print("sqlite3, json, os, datetime")

---

## Ejercicio 1: Comparación de formatos de almacenamiento (CE a, e)

### Escenario
Una empresa de e-commerce tiene un dataset de transacciones con 15 columnas.
Las consultas analíticas típicas solo acceden a 3-4 columnas a la vez.

**Tarea:** Simula datos y compara el tamaño de almacenamiento en CSV vs JSON.
Después, responde las preguntas teóricas.

In [ ]:
np.random.seed(42)
N = 10000

df_transacciones = pd.DataFrame({
    'id_transaccion': range(1, N + 1),
    'fecha': pd.date_range('2025-01-01', periods=N, freq='h'),
    'id_cliente': np.random.randint(1000, 9999, N),
    'id_producto': np.random.randint(100, 999, N),
    'categoria': np.random.choice(['Electrónica', 'Ropa', 'Hogar', 'Alimentación', 'Deportes'], N),
    'cantidad': np.random.randint(1, 10, N),
    'precio_unitario': np.round(np.random.uniform(5, 500, N), 2),
    'descuento_pct': np.random.choice([0, 5, 10, 15, 20], N),
    'metodo_pago': np.random.choice(['Tarjeta', 'Transferencia', 'PayPal', 'Efectivo'], N),
    'region': np.random.choice(['Norte', 'Sur', 'Este', 'Oeste', 'Centro'], N),
    'ciudad': np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao',
                                 'Zaragoza', 'Málaga', 'Murcia'], N),
    'canal': np.random.choice(['Web', 'Móvil', 'Tienda'], N),
    'estado': np.random.choice(['Completada', 'Pendiente', 'Cancelada', 'Devuelta'], N,
                                p=[0.7, 0.1, 0.1, 0.1]),
    'peso_kg': np.round(np.random.uniform(0.1, 30, N), 2),
    'valoracion': np.random.choice([None, 1, 2, 3, 4, 5], N, p=[0.3, 0.05, 0.05, 0.15, 0.25, 0.2])
})

df_transacciones['total'] = np.round(
    df_transacciones['cantidad'] * df_transacciones['precio_unitario'] *
    (1 - df_transacciones['descuento_pct'] / 100), 2
)

print(f"Dataset generado: {len(df_transacciones):,} registros x {len(df_transacciones.columns)} columnas")
print(f"\nColumnas: {list(df_transacciones.columns)}")
df_transacciones.head()

In [ ]:
csv_path = '/tmp/transacciones.csv'
json_path = '/tmp/transacciones.json'

df_transacciones.to_csv(csv_path, index=False)
df_transacciones.to_json(json_path, orient='records', date_format='iso')

csv_size = os.path.getsize(csv_path)
json_size = os.path.getsize(json_path)

print("=" * 50)
print("COMPARACIÓN DE FORMATOS DE ALMACENAMIENTO")
print("=" * 50)
print(f"\nCSV:  {csv_size:>10,} bytes ({csv_size / 1024:.1f} KB)")
print(f"JSON: {json_size:>10,} bytes ({json_size / 1024:.1f} KB)")
print(f"\nJSON es {json_size / csv_size:.1f}x más grande que CSV")

print("\n" + "=" * 50)
print("PREGUNTAS PARA REFLEXIONAR:")
print("=" * 50)
print("""
1. ¿Por qué JSON ocupa más espacio que CSV?
   → JSON repite los nombres de columna en cada registro

2. ¿Qué formato sería mejor para consultas que solo leen 3 columnas?
   → Parquet (columnar): solo lee las columnas necesarias

3. ¿Y si los datos llegan en streaming desde una API?
   → Avro: compacto, soporta evolución de esquema

4. ¿Cuál de estos formatos soporta compresión nativa?
   → Parquet (Snappy, Gzip), Avro (Snappy, Deflate), ORC (Zlib, Snappy)
""")

t_csv = time.time()
_ = pd.read_csv(csv_path, usecols=['categoria', 'total', 'region'])
t_csv = time.time() - t_csv

t_json = time.time()
_ = pd.read_json(json_path)
t_json = time.time() - t_json

print(f"Tiempo lectura CSV (3 cols): {t_csv:.4f}s")
print(f"Tiempo lectura JSON (todas): {t_json:.4f}s")
print("\n💡 Con Parquet, la lectura de 3 columnas sería aún más rápida")
print("   porque el formato columnar solo lee los bytes necesarios.")

---

## Ejercicio 2: Operaciones SQL - Simulando Sqoop (CE b, c)

### Escenario
Simula una base de datos relacional con SQLite y practica las operaciones
que Sqoop realizaría al importar/exportar datos.

In [ ]:
conn = sqlite3.connect(':memory:')

df_transacciones.to_sql('transacciones', conn, index=False, if_exists='replace')

df_clientes = pd.DataFrame({
    'id_cliente': range(1000, 10000),
    'nombre': [f'Cliente_{i}' for i in range(1000, 10000)],
    'email': [f'cliente{i}@email.com' for i in range(1000, 10000)],
    'fecha_registro': pd.date_range('2020-01-01', periods=9000, freq='D')[:9000],
    'segmento': np.random.choice(['Premium', 'Standard', 'Basic'], 9000, p=[0.2, 0.5, 0.3])
})
df_clientes.to_sql('clientes', conn, index=False, if_exists='replace')

print("=== Base de datos creada ===")
print(f"Tabla 'transacciones': {len(df_transacciones):,} registros")
print(f"Tabla 'clientes': {len(df_clientes):,} registros")

print("\n--- SIMULACIÓN: sqoop list-tables ---")
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for t in tablas['name']:
    print(f"  {t}")

In [ ]:
print("=" * 60)
print("EJERCICIO: Escribe las consultas SQL equivalentes")
print("a las operaciones que haría Sqoop")
print("=" * 60)

print("\n--- Consulta 1: Total de ventas por categoría ---")
q1 = pd.read_sql("""
    SELECT categoria,
           COUNT(*) as num_transacciones,
           ROUND(SUM(total), 2) as total_ventas,
           ROUND(AVG(total), 2) as ticket_medio
    FROM transacciones
    WHERE estado = 'Completada'
    GROUP BY categoria
    ORDER BY total_ventas DESC
""", conn)
print(q1.to_string(index=False))

print("\n--- Consulta 2: JOIN transacciones + clientes (simula Sqoop free-form query) ---")
q2 = pd.read_sql("""
    SELECT c.segmento,
           COUNT(DISTINCT t.id_cliente) as clientes_unicos,
           COUNT(*) as transacciones,
           ROUND(SUM(t.total), 2) as total_gastado,
           ROUND(AVG(t.total), 2) as gasto_medio
    FROM transacciones t
    JOIN clientes c ON t.id_cliente = c.id_cliente
    WHERE t.estado = 'Completada'
    GROUP BY c.segmento
    ORDER BY total_gastado DESC
""", conn)
print(q2.to_string(index=False))

print("\n--- Consulta 3: Datos incrementales (simula --incremental append) ---")
ultimo_id_importado = 9500
q3 = pd.read_sql(f"""
    SELECT COUNT(*) as nuevos_registros
    FROM transacciones
    WHERE id_transaccion > {ultimo_id_importado}
""", conn)
print(f"Registros nuevos desde id {ultimo_id_importado}: {q3['nuevos_registros'].iloc[0]}")
print("\n💡 Esto es lo que hace 'sqoop import --incremental append --last-value 9500'")

---

## Ejercicio 3: Pipeline de integración de datos (CE c, d)

### Escenario
Tienes dos fuentes de datos:
1. **Sensores IoT** en formato JSON (simulando datos que llegarían por Flume)
2. **Inventario** en SQL (simulando datos que importarías con Sqoop)

Debes integrarlos, limpiar y generar un informe.

In [ ]:
np.random.seed(123)

sensores_data = []
for i in range(500):
    sensor = {
        'sensor_id': f'SENSOR-{np.random.randint(1, 20):03d}',
        'timestamp': (datetime(2026, 3, 1) + timedelta(minutes=i * 15)).isoformat(),
        'temperatura': round(np.random.normal(22, 5), 1) if np.random.random() > 0.05 else None,
        'humedad': round(np.random.uniform(30, 80), 1) if np.random.random() > 0.03 else None,
        'presion': round(np.random.normal(1013, 10), 1),
        'ubicacion_id': np.random.randint(1, 8)
    }
    sensores_data.append(sensor)

json_str = json.dumps(sensores_data, indent=2)

df_inventario = pd.DataFrame({
    'ubicacion_id': range(1, 8),
    'nombre_ubicacion': ['Almacén A', 'Almacén B', 'Oficina Central',
                          'Sala Servidores', 'Laboratorio', 'Exterior', 'Planta Baja'],
    'temp_min': [15, 15, 18, 16, 18, -10, 18],
    'temp_max': [30, 30, 26, 24, 28, 45, 26],
    'activo': [True, True, True, True, True, True, False]
})

print("=== FUENTES DE DATOS ===")
print(f"\n1. Datos IoT (JSON): {len(sensores_data)} registros")
print(f"   Ejemplo: {json.dumps(sensores_data[0], indent=2)}")
print(f"\n2. Inventario ubicaciones (SQL):")
print(df_inventario.to_string(index=False))

In [ ]:
print("=" * 60)
print("PIPELINE DE INTEGRACIÓN")
print("=" * 60)

# Paso 1: Cargar JSON
df_sensores = pd.DataFrame(json.loads(json_str))
df_sensores['timestamp'] = pd.to_datetime(df_sensores['timestamp'])
print(f"\n✅ Paso 1 - Carga JSON: {len(df_sensores)} registros")
print(f"   Nulos en temperatura: {df_sensores['temperatura'].isna().sum()}")
print(f"   Nulos en humedad: {df_sensores['humedad'].isna().sum()}")

# Paso 2: Limpieza
antes = len(df_sensores)
df_sensores['temperatura'] = df_sensores.groupby('sensor_id')['temperatura'].transform(
    lambda x: x.fillna(x.mean())
)
df_sensores['humedad'] = df_sensores.groupby('sensor_id')['humedad'].transform(
    lambda x: x.fillna(x.mean())
)
print(f"\n✅ Paso 2 - Limpieza: nulos rellenados con media por sensor")
print(f"   Nulos restantes: {df_sensores.isna().sum().sum()}")

# Paso 3: Integración (JOIN)
df_integrado = df_sensores.merge(df_inventario, on='ubicacion_id', how='left')
print(f"\n✅ Paso 3 - Integración (JOIN): {len(df_integrado)} registros")
print(f"   Columnas: {list(df_integrado.columns)}")

# Paso 4: Enriquecimiento
df_integrado['fuera_rango'] = (
    (df_integrado['temperatura'] < df_integrado['temp_min']) |
    (df_integrado['temperatura'] > df_integrado['temp_max'])
)
df_integrado['hora'] = df_integrado['timestamp'].dt.hour
df_integrado['dia_semana'] = df_integrado['timestamp'].dt.day_name()

alertas = df_integrado['fuera_rango'].sum()
print(f"\n✅ Paso 4 - Enriquecimiento: {alertas} lecturas fuera de rango")

# Paso 5: Resumen
resumen = df_integrado.groupby('nombre_ubicacion').agg({
    'temperatura': ['mean', 'min', 'max', 'std'],
    'humedad': ['mean', 'min', 'max'],
    'fuera_rango': 'sum',
    'sensor_id': 'nunique'
}).round(1)

resumen.columns = ['temp_media', 'temp_min', 'temp_max', 'temp_std',
                    'hum_media', 'hum_min', 'hum_max',
                    'alertas', 'sensores']

print(f"\n✅ Paso 5 - Resumen por ubicación:")
print(resumen.to_string())

print("\n" + "=" * 60)
print("PIPELINE COMPLETADO")
print("=" * 60)
print("\n💡 En producción:")
print("   - JSON llegaría por Flume desde los sensores")
print("   - Inventario se importaría con Sqoop desde MySQL")
print("   - El resultado se guardaría en Parquet en HDFS")
print("   - El resumen se exportaría con Sqoop a una BD de reporting")

---

## Ejercicio 4: R vs Python - Comparación práctica (CE d)

Aunque el examen se hace en Python, es importante conocer las equivalencias con R.

In [ ]:
comparacion = {
    'Operación': [
        'Leer CSV',
        'Ver primeras filas',
        'Filtrar filas',
        'Seleccionar columnas',
        'Agrupar y agregar',
        'Ordenar',
        'JOIN',
        'Crear columna',
        'Valores nulos',
        'Gráfico de barras',
    ],
    'Python (pandas)': [
        "pd.read_csv('file.csv')",
        'df.head()',
        "df[df['col'] > 10]",
        "df[['col1', 'col2']]",
        "df.groupby('g')['v'].sum()",
        "df.sort_values('col')",
        "pd.merge(a, b, on='key')",
        "df['new'] = df['a'] + df['b']",
        'df.fillna(0)',
        "df.plot(kind='bar')",
    ],
    'R (tidyverse)': [
        "read_csv('file.csv')",
        'head(df)',
        'df %>% filter(col > 10)',
        'df %>% select(col1, col2)',
        'df %>% group_by(g) %>% summarise(sum(v))',
        'df %>% arrange(col)',
        "inner_join(a, b, by='key')",
        'df %>% mutate(new = a + b)',
        'df %>% replace_na(list(col=0))',
        'ggplot(df, aes(x, y)) + geom_bar()',
    ]
}

df_comp = pd.DataFrame(comparacion)

print("=" * 90)
print("EQUIVALENCIAS PYTHON (pandas) vs R (tidyverse)")
print("=" * 90)
print()
for _, row in df_comp.iterrows():
    print(f"📌 {row['Operación']}")
    print(f"   Python: {row['Python (pandas)']}")
    print(f"   R:      {row['R (tidyverse)']}")
    print()

print("\n💡 Pregunta de examen típica:")
print('   "¿Cuándo elegirías R sobre Python para un proyecto Big Data?"')
print("   → R cuando el foco es análisis estadístico avanzado y visualización")
print("   → Python cuando el foco es ingeniería de datos y producción")

---

## Ejercicio 5: Preguntas tipo examen

Intenta responder cada pregunta antes de ver la solución.

In [ ]:
preguntas = [
    {
        'pregunta': '¿Cuál es la diferencia principal entre Flume y Sqoop?',
        'respuesta': 'Flume ingesta datos en streaming (logs, eventos) hacia HDFS. '
                     'Sqoop transfiere datos por lotes entre RDBMS y HDFS.'
    },
    {
        'pregunta': '¿Por qué Parquet es mejor que CSV para consultas analíticas?',
        'respuesta': 'Parquet es columnar: solo lee las columnas necesarias, '
                     'soporta compresión por columna, y permite predicate pushdown.'
    },
    {
        'pregunta': '¿Qué componentes tiene un agente Flume?',
        'respuesta': 'Source (recibe datos), Channel (buffer temporal), '
                     'Sink (escribe en destino). Ejemplo: Exec Source → Memory Channel → HDFS Sink.'
    },
    {
        'pregunta': '¿Cuál es la diferencia entre ETL y ELT?',
        'respuesta': 'ETL transforma antes de cargar (servidor intermedio). '
                     'ELT carga datos crudos y transforma en destino (data lake). '
                     'ELT es más flexible (schema-on-read).'
    },
    {
        'pregunta': '¿Cuándo usarías import incremental en Sqoop?',
        'respuesta': 'Cuando la tabla crece continuamente y no quieres reimportar todo. '
                     'Modo "append" para nuevas filas, "lastmodified" para filas actualizadas.'
    },
    {
        'pregunta': 'Nombra 3 sistemas NoSQL y su modelo de datos.',
        'respuesta': 'HBase: columnar (wide-column). '
                     'MongoDB: documental (JSON/BSON). '
                     'Redis: clave-valor (in-memory). '
                     'Otros: Cassandra (columnar), Elasticsearch (índice invertido).'
    },
]

print("=" * 60)
print("PREGUNTAS DE AUTOEVALUACIÓN")
print("=" * 60)

for i, q in enumerate(preguntas, 1):
    print(f"\n{'─' * 60}")
    print(f"❓ Pregunta {i}: {q['pregunta']}")
    print(f"\n   💡 Respuesta: {q['respuesta']}")

print("\n" + "=" * 60)
print("¿Has acertado todas? Repasa los temas donde tengas dudas.")
print("=" * 60)

---

## Resumen de contenidos evaluables

| Bloque | Temas clave | CE |
|--------|------------|----|
| Almacenamiento | CSV, JSON, Avro, Parquet, ORC, SequenceFile | a, e |
| Sistemas NoSQL | HBase, Cassandra, MongoDB, Redis, Elasticsearch | a |
| Flume | Source, Channel, Sink, configuración de agentes | b |
| Sqoop | Import, Export, incremental, free-form query | b |
| Pipelines | ETL vs ELT, componentes, diseño | c |
| Programación | Python (pandas) vs R (tidyverse) | d |

---

**¡Buena suerte en el examen!** 🍀